In [2]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 54 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (521 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time
import requests

OLLAMA_URL = "http://127.0.0.1:11434"

# First, check whether Ollama is already running
try:
    response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
    response.raise_for_status()
    print("Ollama server is already running!")

except requests.RequestException:
    # Start Ollama only if it is not running
    try:
        ollama_process = subprocess.Popen(
            ["ollama", "serve"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
    except FileNotFoundError:
        raise RuntimeError(
            "Ollama not found. Install Ollama and add it to PATH."
        )

    # Wait for the server to become ready
    for attempt in range(30):
        try:
            response = requests.get(
                f"{OLLAMA_URL}/api/tags",
                timeout=2
            )

            if response.status_code == 200:
                print("Ollama server started successfully!")
                break

        except requests.RequestException:
            time.sleep(2)

    else:
        raise RuntimeError(
            "Ollama server did not start. "
            "Open Ollama manually and try again."
        )

# Show installed models
response = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
models = response.json().get("models", [])

print("\nInstalled models:")
for model in models:
    print("-", model["name"])

Ollama server started successfully!

Installed models:


In [60]:
!ollama pull qwen2.5:3b

In [61]:
!ollama list

NAME                        ID              SIZE      MODIFIED       
qwen2.5:3b                  357c53fb659c    1.9 GB    2 seconds ago     
bangla-translator:latest    ff7154d2c985    1.9 GB    3 minutes ago     
qwen2.5:7b                  845dbda0ea48    4.7 GB    3 minutes ago     
translategemma:12b          c2f9a9ca1ec7    8.1 GB    8 minutes ago     
translategemma:4b           c49d986b0764    3.3 GB    14 minutes ago    


In [62]:
modelfile_content = '''
FROM qwen2.5:3b

SYSTEM """
You are a professional Bengali-English translator.

Rules:
1. If the user provides Bengali text, translate it into natural English.
2. If the user provides English text, translate it into natural Bengali.
3. Preserve the original meaning, tone, and context.
4. Do not explain or summarize.
5. Return only the translated text.
"""

PARAMETER temperature 1
'''

with open("Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile_content.strip() + "\n")

print("Modelfile created successfully!")
print(modelfile_content)

Modelfile created successfully!

FROM qwen2.5:3b

SYSTEM """
You are a professional Bengali-English translator.

Rules:
1. If the user provides Bengali text, translate it into natural English.
2. If the user provides English text, translate it into natural Bengali.
3. Preserve the original meaning, tone, and context.
4. Do not explain or summarize.
5. Return only the translated text.
"""

PARAMETER temperature 1



***Create Custom Bangla-English Translation Model***

In [63]:
import subprocess

result = subprocess.run(
    ["ollama", "create", "bangla-translator", "-f", "Modelfile"],
    capture_output=True,
    text=True
)
# printing error message
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Custom model creation failed.")

print("Custom translation model is ready!")

Custom translation model is ready!


In [57]:
import requests

# ==========================================
# 1. Ollama API Configuration
# ==========================================

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"
MODEL_NAME = "bangla-translator"


# ==========================================
# 2. Translation Function
# ==========================================

def translate_text(text, direction="auto"):

    # Validate input
    if not text or not text.strip():
        return "Please provide some text."

    # Set translation direction
    if direction == "bn-en":
        instruction = "Translate the following Bangla text into English."

    elif direction == "en-bn":
        instruction = "Translate the following English text into Bangla."

    else:
        instruction = (
            "If the text is Bangla, translate it into English. "
            "If the text is English, translate it into Bangla."
        )

    # Prepare the prompt
    prompt = f"""
{instruction}
Return only the translated text.
Do not add explanations.

Text:{text}
"""

    # Send request to Ollama API
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt,
            "stream": False
        },
        timeout=180
    )

    # Handle API response
    response.raise_for_status()

    return response.json()["response"].strip()

print("Bangla-English Translator is ready!")

Bangla-English Translator is ready!


In [64]:
text = input("Enter Bangla or English text: ")

print("\nChoose translation direction:")
print("1. Bangla to English")
print("2. English to Bangla")

choice = input("Enter your choice (1/2): ").strip()

if choice == "1":
    direction = "bn-en"
elif choice == "2":
    direction = "en-bn"
else:
    print("Invalid choice! Please enter 1 or 2.")
    direction = None

if direction:
    try:
        output = translate_text(text, direction)

        print("\nOriginal Text:", text)
        print("Translation:", output)

    except Exception as e:
        print("Error:", e)

Enter Bangla or English text: I live in Bangladesh

Choose translation direction:
1. Bangla to English
2. English to Bangla
Enter your choice (1/2): 2

Original Text: I live in Bangladesh
Translation: আমি বাংলাদেশে বাস করি
